# Lesson 5.1 · The neighbors model

KNN is the simplest model there is and for that very reason it is an excellent tool to understand what a model is: **it learns nothing**. It memorizes the training set and,
for every new point, looks at the `k` nearest examples and reports their most frequent class (or the mean of the target, in regression). "Similar to what I already know" is the only idea.

**What you will learn**

- What `fit` and `predict` do in a model that has no parameters to estimate.
- Why `k` is the complexity regulator (bias and variance) and how it shows on the decision boundary.
- How to read `predict_proba` and `kneighbors`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(n_samples=400, noise=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
print(X_tr.shape, "| classes:", np.bincount(y_tr))

## What "training" a KNN means

`fit` does not compute coefficients: it stores `X_tr` and `y_tr` (and builds, if needed, a structure to search quickly). All the work happens in **`predict`**:
for every test point it computes the distances from all the training examples, takes the `k` smallest and votes.

```
predict(x):  d = distances(x, X_tr) → the k indices with minimum d → most frequent class among their y
```

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
x = X_te[:1]
distances, indices = knn.kneighbors(x)                      # the 5 neighbors of a point, with the distances
print("indices of the neighbors:", indices[0], "| distances:", distances[0].round(3))
print("classes of the neighbors:", y_tr[indices[0]], "-> predict:", knn.predict(x), "| predict_proba:", knn.predict_proba(x))

`predict_proba` is not a calibrated probability: it is the **fraction of neighbors** of each class (with `k = 5` it has only the values 0, 0.2, ..., 1). You come back to it in exercise D2.

## `k` regulates the complexity

- `k = 1`: every point takes the label of the nearest neighbor. The boundary follows every single example, noise included: **overfitting** (low bias, high variance), and on the training set the accuracy is 1.
- `k = n`: everyone predicts the most frequent class: **underfitting**.
- In between the boundary smooths out. Look at it.

In [ ]:
grid1, grid2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
points = np.c_[grid1.ravel(), grid2.ravel()]
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for a, k in zip(ax, [1, 15, 100]):
    m = KNeighborsClassifier(k).fit(X_tr, y_tr)
    a.contourf(grid1, grid2, m.predict(points).reshape(grid1.shape), alpha=0.25, cmap="coolwarm")
    a.scatter(*X_tr.T, c=y_tr, s=12, cmap="coolwarm", edgecolor="k", linewidth=0.3)
    a.set_title(f"k = {k}: train {m.score(X_tr, y_tr):.2f} | test {m.score(X_te, y_te):.2f}")
plt.tight_layout(); plt.show()

With `k = 1` the boundary is jagged and the training accuracy is 1.00, but the test one is lower. With `k = 100` the boundary is almost a line and the model underestimates the
curvature. The compromise is found with cross-validation, like for every hyperparameter (lesson 5.3).

**Watch out for a detail.** With `k = 1` the prediction on a **training** point is the label of the point itself (the nearest neighbor of a training point is itself, distance 0): that is why
the training accuracy is 1 and says nothing. Whoever evaluates a KNN on the data it was trained on gets a meaningless number (exercise E1).

## Weights and voting

By default every neighbor weighs the same (`weights="uniform"`). With `weights="distance"` every neighbor weighs `1/distance`: the really close neighbors count more and ties are almost always broken. For the same reason as before, with the weights the training point has distance 0 from itself and weighs infinitely: the **training** accuracy with `weights="distance"` is always 1, whatever `k` is (look at the table below).

In [ ]:
for w in ["uniform", "distance"]:
    m = KNeighborsClassifier(15, weights=w).fit(X_tr, y_tr)
    print(f"{w:9s} | train {m.score(X_tr, y_tr):.3f} | test {m.score(X_te, y_te):.3f}")

> **Common mistake.** Thinking that a KNN "without training" cannot overfit. KNN with small `k` is one of the models most prone to overfitting: all the complexity sits in `k`.

## Check your understanding

<details><summary>Why is the accuracy on the training set always 1 with `k=1` (without duplicates with different labels)?</summary>

Because the nearest neighbor of every training point is the point itself, at distance 0, and its label is the true one. To evaluate a KNN you need a test set or cross-validation, never the training set.

</details>

<details><summary>If the two sides of a tie have the same number of neighbors, what does scikit-learn do?</summary>

With `weights="uniform"` it chooses the class with the smallest label (the tie is resolved deterministically but arbitrarily). That is why with two classes odd `k` are preferred, and with more classes `weights="distance"` is better.

</details>